# Project 4 — Notebook 15: Business Summary & Recommendations
### City-Level Intelligence — Targeted Interventions by City Tier

---

| | |
|---|---|
| **Scope** | NCR (Region 3) · City-level performance decomposition |
| **Feeds from** | NB13 (City Performance Landscape) · NB14 (City Deep Dive) |
| **Audience** | Operations Leadership / Area Heads |

---

### What Project 4 Investigated

| Question | Notebook | Status |
|----------|----------|--------|
| Which cities drive zone-level ticket volume and underperformance? | NB13 | ✅ |
| Is zone-level underperformance concentrated or broadly distributed? | NB13 | ✅ |
| Which cities are driving the P3.2 degradation breach problem? | NB14 | ✅ |
| Does endorsement delay cluster in specific cities? | NB14 | ✅ |
| Is Zone 5's elevated field time concentrated in CBD cities? | NB14 | ✅ |
| Which cities have the worst UNKNOWN ticket rates? | NB14 | ✅ |

## 1. Setup

In [1]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline

os.chdir(os.path.join('..', '..'))
if os.path.abspath(os.getcwd()) not in sys.path:
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from src.fault_ticket.metrics import calculate_zone_summary
from config import ZONE_ORDER, ZONE_PALETTE

df      = pd.read_csv('output/cleaned_fault_ticket.csv')
df_zone = df[(df['ZONE'].isin(ZONE_ORDER)) & (df['Priority'] < 4)].copy()
clean   = df_zone[df_zone['Timestamp_Integrity']].copy()

MIN_CITY_TICKETS = 50
city_vol     = df_zone.groupby(['ZONE','CITY']).size().reset_index(name='Tickets')
valid_cities = city_vol[city_vol['Tickets'] >= MIN_CITY_TICKETS][['ZONE','CITY']]
df_city      = df_zone.merge(valid_cities, on=['ZONE','CITY'])
clean_city   = clean.merge(valid_cities, on=['ZONE','CITY'])

# Rebuild city summary (same logic as NB13)
city_summary = (df_city
    .groupby(['ZONE','CITY'])
    .agg(
        Tickets      = ('SLA_Compliant', 'count'),
        SLA_pct      = ('SLA_Compliant', 'mean'),
        Avg_MTTR     = ('OUTAGEDURATION', 'mean'),
        Unique_Sites = ('SiteName',       'nunique'),
    ).reset_index()
)
city_summary['SLA_pct']      *= 100
city_summary['Fault_Density'] = city_summary['Tickets'] / city_summary['Unique_Sites']
zone_sla = df_zone.groupby('ZONE')['SLA_Compliant'].mean().mul(100)
city_summary['Zone_SLA']    = city_summary['ZONE'].map(zone_sla)
city_summary['SLA_vs_Zone'] = city_summary['SLA_pct'] - city_summary['Zone_SLA']

noc_ft = clean_city.groupby(['ZONE','CITY']).agg(
    Avg_NOC_Time   = ('DISPATCH_DELAY_HOURS', 'mean'),
    Avg_Field_Time = ('FIELD_TIME_HOURS',     'mean'),
).reset_index()
city_summary = city_summary.merge(noc_ft, on=['ZONE','CITY'], how='left')

# P3.2 breach by city
p32 = df_city[df_city['Priority_Urgency'] == 3.2]
p32_city = (p32.groupby(['ZONE','CITY'])
    .agg(P32_Tickets=('SLA_Compliant','count'),
         P32_Breach =('SLA_Compliant', lambda x:(1-x.mean())*100))
    .reset_index().query('P32_Tickets >= 20'))

# UNKNOWN rate by city
unk_city = (df_city.groupby(['ZONE','CITY'])
    .agg(Total=('SLA_Compliant','count'),
         Unknown=('Standardized RFO', lambda x:(x=='UNKNOWN-Under Investigation').sum()))
    .reset_index())
unk_city['Unknown_Rate'] = unk_city['Unknown'] / unk_city['Total'] * 100

# Composite city risk: normalised rank average of breach + density + unknown rate
city_risk = city_summary.merge(
    p32_city[['ZONE','CITY','P32_Breach']], on=['ZONE','CITY'], how='left'
).merge(
    unk_city[['ZONE','CITY','Unknown_Rate']], on=['ZONE','CITY'], how='left'
)
city_risk['Breach_Rate'] = 100 - city_risk['SLA_pct']
for col in ['Breach_Rate','Fault_Density','Unknown_Rate','P32_Breach']:
    mn, mx = city_risk[col].min(), city_risk[col].max()
    city_risk[f'{col}_norm'] = (city_risk[col] - mn) / (mx - mn + 1e-9) * 100
city_risk['Risk_Score'] = (
    city_risk['Breach_Rate_norm']  * 0.35 +
    city_risk['P32_Breach_norm']   * 0.30 +
    city_risk['Fault_Density_norm']* 0.20 +
    city_risk['Unknown_Rate_norm'] * 0.15
)
city_risk = city_risk.sort_values('Risk_Score', ascending=False).reset_index(drop=True)
city_risk['Rank'] = range(1, len(city_risk) + 1)

# Zone-level stats for narrative
top_zone    = zone_sla.idxmax()
bottom_zone = zone_sla.idxmin()
top_city    = city_risk.iloc[0]

print(f"✅ {len(df_city):,} city-filtered reactive tickets")
print(f"   Cities analysed: {len(city_summary)}")
print(f"   Top risk city: {top_city['CITY']} ({top_city['ZONE']}) — score {top_city['Risk_Score']:.0f}")

✅ 36,847 city-filtered reactive tickets
   Cities analysed: 17
   Top risk city: PARANAQUE CITY (ZONE 6) — score 62


## 2. Key Metrics Snapshot

In [2]:
worst_sla_city  = city_summary.loc[city_summary['SLA_pct'].idxmin()]
best_sla_city   = city_summary.loc[city_summary['SLA_pct'].idxmax()]
worst_mttr_city = city_summary.loc[city_summary['Avg_MTTR'].idxmax()]
worst_density   = city_summary.loc[city_summary['Fault_Density'].idxmax()]
worst_p32_city  = p32_city.loc[p32_city['P32_Breach'].idxmax()] if len(p32_city) else None
worst_unk_city  = unk_city.loc[unk_city['Unknown_Rate'].idxmax()]
worst_noc_city  = city_summary.loc[city_summary['Avg_NOC_Time'].idxmax()]

ncr_unk_rate    = len(df_city[df_city['Standardized RFO']=='UNKNOWN-Under Investigation']) / len(df_city) * 100

metrics = pd.DataFrame({
    'Metric': [
        'Highest-risk city (composite score)',
        'Worst SLA compliance (city)',
        'Best SLA compliance (city)',
        'Longest avg MTTR (city)',
        'Highest fault density (city)',
        'Highest P3.2 breach rate (city)',
        'Highest endorsement delay (city)',
        'Highest UNKNOWN ticket rate (city)',
        'NCR-wide UNKNOWN ticket rate',
    ],
    'Value': [
        f"{top_city['CITY']}  ({top_city['ZONE']} · risk score {top_city['Risk_Score']:.0f}/100)",
        f"{worst_sla_city['CITY']}  ({worst_sla_city['ZONE']} · {worst_sla_city['SLA_pct']:.1f}%)",
        f"{best_sla_city['CITY']}  ({best_sla_city['ZONE']} · {best_sla_city['SLA_pct']:.1f}%)",
        f"{worst_mttr_city['CITY']}  ({worst_mttr_city['ZONE']} · {worst_mttr_city['Avg_MTTR']:.0f}h)",
        f"{worst_density['CITY']}  ({worst_density['ZONE']} · {worst_density['Fault_Density']:.1f} faults/site)",
        f"{worst_p32_city['CITY']}  ({worst_p32_city['ZONE']} · {worst_p32_city['P32_Breach']:.1f}% breach)" if worst_p32_city is not None else 'N/A',
        f"{worst_noc_city['CITY']}  ({worst_noc_city['ZONE']} · {worst_noc_city['Avg_NOC_Time']:.1f}h avg)",
        f"{worst_unk_city['CITY']}  ({worst_unk_city['ZONE']} · {worst_unk_city['Unknown_Rate']:.1f}%)",
        f"{ncr_unk_rate:.1f}%  (5,129 tickets)",
    ]
})

display(metrics.style
    .set_properties(**{'text-align':'left'})
    .set_table_styles([{'selector':'th','props':[('text-align','left')]}])
    .hide(axis='index')
)

Metric,Value
Highest-risk city (composite score),PARANAQUE CITY (ZONE 6 · risk score 62/100)
Worst SLA compliance (city),RIZAL (ZONE 3 · 78.8%)
Best SLA compliance (city),SAN JUAN CITY (ZONE 4 · 88.3%)
Longest avg MTTR (city),LAS PINAS CITY (ZONE 6 · 112h)
Highest fault density (city),SAN JUAN CITY (ZONE 4 · 19.3 faults/site)
Highest P3.2 breach rate (city),PARANAQUE CITY (ZONE 6 · 50.0% breach)
Highest endorsement delay (city),SAN JUAN CITY (ZONE 4 · 38.2h avg)
Highest UNKNOWN ticket rate (city),SAN JUAN CITY (ZONE 4 · 63.7%)
NCR-wide UNKNOWN ticket rate,"13.9% (5,129 tickets)"


## 3. Key Findings

### Finding 1 🟠 — Zone Underperformance Is City-Concentrated, Not Broadly Distributed

Zone-level SLA and MTTR averages mask significant within-zone variation. In most zones, one or two cities are clear underperformers relative to the zone average — meaning zone-level improvement efforts need city-level targeting to be efficient. Broad-zone interventions (e.g., additional technicians deployed zone-wide) will have lower impact per unit cost than city-specific protocols addressing the specific friction type (fault recurrence, data quality gaps, or field time elevation).

### Finding 2 🔴 — P3.2 Breach Concentration Points to Specific Cities

The P3.2 degradation breach problem identified in Project 3 (31.6–44.2% zone-level) is not uniformly distributed within zones. Specific cities consistently exceed their zone's P3.2 breach average. Because P3.2 carries a 12h SLA and tickets in these cities show elevated DISPATCH_DELAY_HOURS, a material portion of the SLA window is consumed before field dispatch begins. The data does not isolate the cause of the endorsement delay — it may reflect contact chain length, scheduling patterns, or NOC triage time. The fix for these cities is a maximum endorsement handoff SLA for P3.2 tickets, enforced and visible to the area head, regardless of cause.

### Finding 3 🔴 — Zone 5 CBD Field Time is Geographically Concentrated

Zone 5's elevated field time is not distributed evenly across the zone. CBD cities (Makati, Taguig/BGC) show materially higher FIELD_TIME_HOURS than non-CBD cities, confirming that the friction is geographically driven rather than a zone-wide capacity problem. The most operationally plausible explanation is building work permit friction in high-density CBD sites — a known constraint in Makati and BGC — but the data does not separately measure permit wait time from repair time. The intervention implication holds regardless: CBD-site protocols (pre-coordinated access, dedicated permit teams) are more efficient than general technician deployment.

### Finding 4 🟡 — UNKNOWN Ticket Rates Cluster in Specific Cities

The 13.9% NCR-wide UNKNOWN-Under Investigation RFO rate is not evenly distributed.
Cities with the highest UNKNOWN rates have the least reliable RFO data, making
fault-type analysis and intervention prioritisation less accurate for those cities.

The most likely mechanism: a network alarm clears and NOC/ROC tags the ticket as
'Under Investigation'. The field engineer is dispatched, conducts an on-site
investigation, and files a report — but no RFO update is made to the ticket before
the WO is closed. Ticket closure is controlled by the field engineer (not the team
lead), so the accountability gap is at the engineer level: the WO is closed once the
alarm is confirmed cleared, but the RFO field is left unpopulated.

The city clustering pattern likely reflects which areas have the highest rate of
alarm-cleared-without-confirmed-cause faults (e.g. intermittent power or
self-recovering transmission issues where the root cause is genuinely uncertain
at time of close) rather than deliberate omission. This is a data capture workflow
issue — the WO close step should require a non-UNKNOWN RFO selection, or flag
tickets closed with UNKNOWN status for NOC/ROC follow-up within 24h.

## 4. City Risk Scorecard — Priority Intervention Targets

> Composite city risk score = weighted combination of:
> SLA breach rate (35%) · P3.2 breach rate (30%) · fault density (20%) · UNKNOWN rate (15%)
> Higher score = higher intervention priority.

In [3]:
# Display top 15 city risk scorecard
display_cols = ['Rank','ZONE','CITY','Tickets','SLA_pct','P32_Breach',
                'Fault_Density','Unknown_Rate','Avg_NOC_Time','Risk_Score']
score_display = city_risk[display_cols].head(10).copy()
score_display = score_display.rename(columns={
    'SLA_pct'      : 'SLA%',
    'P32_Breach'   : 'P3.2 Breach%',
    'Fault_Density': 'Density',
    'Unknown_Rate' : 'UNKNOWN%',
    'Avg_NOC_Time' : 'NOC_h',
    'Risk_Score'   : 'Score',
})

def color_risk(val):
    if isinstance(val, float):
        if val >= 70: return 'background-color: #fdecea'
        if val >= 50: return 'background-color: #fef9e7'
    return ''

display(score_display.style
    .format({
        'SLA%':'      {:.1f}%',
        'P3.2 Breach%':'{:.1f}%',
        'Density':     '{:.1f}',
        'UNKNOWN%':    '{:.1f}%',
        'NOC_h':       '{:.1f}h',
        'Score':       '{:.0f}',
    })
    .map(color_risk, subset=['Score'])
    .set_caption('City Risk Scorecard — Top 10 Priority Targets')
    .hide(axis='index')
)

Rank,ZONE,CITY,Tickets,SLA%,P3.2 Breach%,Density,UNKNOWN%,NOC_h,Score
1,ZONE 6,PARANAQUE CITY,1775,81.4%,50.0%,8.5,8.6%,0.1h,62
2,ZONE 5,PASAY CITY,1601,80.0%,44.2%,9.9,11.3%,0.1h,61
3,ZONE 6,LAS PINAS CITY,996,78.8%,41.2%,8.2,15.9%,0.1h,59
4,ZONE 5,TAGUIG CITY,2342,80.4%,44.5%,8.1,7.7%,0.1h,56
5,ZONE 3,RIZAL,2723,78.8%,43.3%,5.8,0.8%,0.1h,55
6,ZONE 3,MALABON CITY,974,81.3%,41.1%,11.3,14.6%,0.3h,54
7,ZONE 5,MAKATI CITY,5145,79.2%,34.6%,8.6,26.3%,0.1h,51
8,ZONE 4,SAN JUAN CITY,1451,88.3%,40.5%,19.3,63.7%,38.2h,51
9,ZONE 1,VALENZUELA CITY,1845,83.7%,40.4%,9.5,16.0%,0.1h,41
10,ZONE 1,CALOOCAN CITY,2282,82.5%,37.1%,9.6,14.1%,30.6h,41


## 5. Recommendations

### Immediate (0–3 Months)

**REC-P4-01 · Enforce P3.2 Endorsement Handoff SLA for Top Breach Cities**

NB14 identifies specific cities where P3.2 degradation breach rates consistently exceed their zone average, and where DISPATCH_DELAY_HOURS is also elevated relative to the zone. Because P3.2 carries only a 12h SLA, a long NOC-to-dispatch window directly compresses the time available for field resolution. The data does not isolate the specific cause of the endorsement delay — it may reflect contact chain length, scheduling patterns, or NOC triage time. Set a maximum 2h endorsement-to-dispatch SLA for P3.2 tickets specifically. The duty
team lead is accountable; the area head reviews P3.2 breach exceptions weekly for
the flagged cities until breach rate falls within 5pp of zone average.

**REC-P4-02 · Zone 5 CBD Sites — Duty Engineer Pre-Brief and Access Protocol**

NB14 confirms Zone 5's elevated field time is geographically concentrated in CBD cities (Makati, Taguig/BGC), not broadly distributed across the zone. The most operationally plausible explanation is building access friction at commercial sites — a known constraint in high-density CBD areas. The data does not separately measure permit wait time from repair time, but the geographic concentration makes a broad technician deployment inefficient. The intervention is protocol: for every CBD site in Makati and Taguig/BGC, the team lead must maintain an up-to-date building access file (permit contact, access procedure, building admin number) accessible to the
duty team, not only the assigned engineer. Day-off endorsements for CBD sites must
include an explicit access handoff — the duty engineer should not arrive at a commercial
building without prior coordination.

---

### Short-Term (3–6 Months)

**REC-P4-03 · Address High-Risk Cities from City Risk Scorecard**

The top 10 cities by composite risk score (NB15 scorecard) should each have a named
action owner — either the team lead for that city or the area head — with a quarterly
KPI target for SLA compliance improvement. Cities in the high breach + high density
quadrant (NB14 risk matrix) are candidates for site-level PM scheduling (Project 5).
Cities in the high breach + low density quadrant are endorsement or capacity problems
requiring protocol review, not PM.

**REC-P4-04 · Require RFO Selection at WO Close Step for High-UNKNOWN Cities**

Cities with UNKNOWN-Under Investigation rates materially above the NCR avg (13.9%)
have unreliable RFO data. The gap is at the field engineer level: WOs are being
closed once the alarm clears without a confirmed RFO being entered. The fix is a
system-level enforcement — the WO close workflow should require selection of a
non-UNKNOWN RFO, or automatically flag tickets closed with UNKNOWN status for
NOC/ROC follow-up within 24h. Short of a system change, NOC/ROC should issue a
standing instruction: any ticket closed with UNKNOWN RFO is re-opened for engineer
clarification within the same shift. Track UNKNOWN rate monthly per city —
target: below 10% for all cities within 6 months.

---

### Strategic (6–12 Months)

**REC-P4-05 · Proceed to Project 5 — Site-Level Risk Profiling**
Project 4 has identified which cities contain the highest-risk concentration.
Project 5 will drill to the site level within those cities: identify the specific
sites generating disproportionate fault volume, build a site risk score, and generate
a prioritised PM schedule for area heads and team leads.

## 6. Next Steps

| Priority | Action | Owner | Timeline |
|----------|--------|-------|----------|
| 🔴 High | P3.2 endorsement handoff SLA — top breach cities (NB14 Section 2) | Area Heads | Month 1 |
| 🔴 High | Zone 5 CBD access protocol — duty engineer pre-brief per site | Zone 5 Area Head | Month 1–2 |
| 🟡 Medium | City risk scorecard (NB15) — assign named owner per top-10 city | Area Heads | Month 2 |
| 🟡 Medium | UNKNOWN RFO rate — mandate closure confirmation for flagged cities | Team Leads | Month 2–3 |
| 🟢 Standard | Cross-zone knowledge transfer — Zone 4 city-level protocols documented | Ops Leadership | Month 3–4 |
| ▶ Next | **Commence Project 5 — Site-Level Risk Profiling** | Analytics | Month 2 |

---

> **Project 5 will answer:** Within the high-risk cities identified in Project 4,
> which specific sites are generating disproportionate fault volume? What is each
> site's risk profile across fault frequency, MTTR, SLA breach rate, and RFO diversity?
> What is the prioritised PM schedule for each zone's area head?